# BiLSTM for Protein Secondary Structure Prediction

This notebook implements a Bidirectional LSTM approach for predicting protein secondary structure (helix, sheet, or coil).

## Key Differences from MLP

| Aspect | MLP | BiLSTM |
|--------|-----|--------|
| Representation | One-hot (273 dim) | Embedding (dense) |
| Context | Fixed window | Full sequences |
| Direction | Single | Bidirectional |
| Memory | No memory | Long-term dependencies |

In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
from sklearn.model_selection import train_test_split
import time
import random
import copy
import gc

## Data Loading Functions

In [2]:
def get_data(data_name):
    with open(data_name,'r') as data:
        return data.read()

AA_MAP = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4, 'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9, 
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14, 'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '<PAD>': 20
}

LABEL_MAP = {'h': 0, 'e': 1, '_': 2}

In [3]:
def parse_raw_data(raw_text):
    proteins = []
    current_aa_seq = []
    current_label_seq = []
    
    lines = raw_text.strip().split('\n')
    
    for line in lines:
        line = line.strip()
        if not line: continue
        
        if line == '<>':
            if current_aa_seq:
                proteins.append((current_aa_seq, current_label_seq))
            current_aa_seq = []
            current_label_seq = []
        else:
            parts = line.split()
            if len(parts) == 2:
                aa, label = parts[0], parts[1]
                if aa in AA_MAP:
                    current_aa_seq.append(AA_MAP[aa])
                    current_label_seq.append(LABEL_MAP[label])
    
    if current_aa_seq:
        proteins.append((current_aa_seq, current_label_seq))
    
    return proteins

In [4]:
def create_windows(proteins, window_size=13):
    half_window = window_size // 2
    X = []
    Y = []
    
    for aa_seq, label_seq in proteins:
        L = len(aa_seq)
        padded_seq = [AA_MAP['<PAD>']] * half_window + aa_seq + [AA_MAP['<PAD>']] * half_window
        
        for i in range(L):
            window = padded_seq[i : i + window_size]
            target = label_seq[i]
            X.append(window)
            Y.append(target)
    
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

In [5]:
raw_data_sample = get_data('protein-secondary-structure.train')
data_train = parse_raw_data(raw_data_sample)
print(f"Nombre de proteines dans le train: {len(data_train)}")

X_true, y_true = create_windows(data_train, window_size=13)
print(f"Forme X train: {X_true.shape}")

raw_data_sample = get_data('protein-secondary-structure.val')
val_train = parse_raw_data(raw_data_sample)
X_val, y_val = create_windows(val_train, window_size=13)
print(f"Nombre de proteines en validation: {len(val_train)}")

raw_data_sample = get_data('protein-secondary-structure.test')
data = parse_raw_data(raw_data_sample)
x_test, y_test = create_windows(data, window_size=13)
print(f"Nombre de proteines en test: {len(data)}")

Nombre de proteines dans le train: 99
Forme X train: torch.Size([16072, 13])
Nombre de proteines en validation: 12
Nombre de proteines en test: 17


In [6]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositif utilise: {device}")

x_test = x_test.to(device)
X_true = X_true.to(device)
X_val = X_val.to(device)

y_test = y_test.to(device)
y_true = y_true.to(device)
y_val = y_val.to(device)

Dispositif utilise: cuda


## BiLSTM Model Architecture

### Architecture

```
Input [Batch, 13]  -->  Embedding [Batch, 13, 42]
                           |
                           v
                      BiLSTM [Batch, 13, 128]
                           |
                           v
               (Extract central state)
                           v
                 FC [Batch, 128] --> [Batch, 3]
```

In [7]:
class ProteinBiLSTM(nn.Module):
    def __init__(self, vocab_size=21, embedding_dim=42, hidden_dim=64, num_layers=2):
        super().__init__()
        
        # Embedding layer: transforms indices (0-20) into dense vectors
        # Instead of one-hot (21 dim), we get dense embeddings (42 dim)
        # The model learns better representations during training
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Bidirectional LSTM
        # bidirectional=True means we read sequence in both directions
        # Forward: start -> end
        # Backward: end -> start
        # This doubles the output: 64 * 2 = 128
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )
        
        # Classification head
        # Input is hidden_dim * 2 = 128 (bidirectional)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 40),
            nn.ReLU(),
            nn.Dropout(0.6),
            
            nn.Linear(40, 3)
        )

    def forward(self, x):
        # Step 1: Embedding
        # Transform indices [Batch, 13] into vectors [Batch, 13, 42]
        x = self.embedding(x)
        
        # Step 2: LSTM
        # Returns all hidden states for each timestep
        # Output shape: [Batch, 13, 128]
        out, (hn, cn) = self.lstm(x)
        
        # Step 3: Pooling (extract central residue)
        # For a window of 13, central index is 6
        # We extract the hidden state at this position
        mid_index = x.shape[1] // 2
        central_features = out[:, mid_index, :]
        
        # Step 4: Classification
        return self.fc(central_features)

In [8]:
seed = 5587
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)


## Training

In [9]:
model = ProteinBiLSTM().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.00188)
loss_fn = nn.CrossEntropyLoss()

epochs = 60

for i in range(epochs):
    model.train()
    
    y_train = model(X_true)
    loss = loss_fn(y_train, y_true)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if i % 50 == 0:
        y_pred = torch.argmax(y_train, dim=1)
        accuracy = (y_pred == y_true).float().mean()
        print(f"Epoch {i:3d} | Loss: {loss.item():.4f} | Acc Train: {accuracy.item()*100:.2f}%")
        
        with torch.no_grad():
            model.eval()
            y_val_train = model(X_val)
            y_val_pred = torch.argmax(y_val_train, dim=1)
            accuracy_val = (y_val_pred == y_val).float().mean()
            print(f"           | Accuracy Val: {accuracy_val*100:.2f}%")
            model.train()

Epoch   0 | Loss: 1.0910 | Acc Train: 40.79%
           | Accuracy Val: 56.27%
Epoch  50 | Loss: 0.8133 | Acc Train: 63.39%
           | Accuracy Val: 65.13%


## Evaluation

In [10]:
model.eval()

with torch.no_grad():
    y_test_train = model(x_test)
    y_test_pred = torch.argmax(y_test_train, dim=1)
    accuracy = (y_test_pred == y_test).float().mean()
    print(f"Accuracy sur le test set (BiLSTM): {accuracy*100:.2f}%")

Accuracy sur le test set (BiLSTM): 64.09%


# Hyperparameter Search: Battle Royale

C’est une méthode pour **trouver les meilleurs hyperparamètres** pour un modèle de réseau de neurones.  

Les hyperparamètres sont des paramètres qu'on règle **avant l’entraînement**, comme :  

- **Learning rate** : vitesse d’apprentissage  
- **Batch size** : taille des lots de données  
- **Seed** : graine pour la reproductibilité  


In [11]:
"""
import time
import random
import copy
import numpy as np
import torch
import torch.nn as nn

# --- CONFIGURATION DE LA BATAILLE ---
NUM_TRIALS = 50          # 50 Combattants
MAX_EPOCHS = 600         
PATIENCE = 30            # Si la validation ne bouge pas pendant 30 tours -> Next

global_best_model_state = None
global_best_params = {}

print(f" DÉBUT DE LA BATTLE ROYALE : {NUM_TRIALS} COMBATTANTS (Objectif: Max Test Score)")
print("="*60)

start_time = time.time()

for trial in range(1, NUM_TRIALS + 1):
    
    # 1. PARAMÈTRES ALÉATOIRES
    seed = random.randint(0, 10000)
    # Learning Rate (Log-Uniform : explore aussi bien 0.0001 que 0.01)
    lr_log = np.random.uniform(-4, -2.3) 
    lr = 10 ** lr_log
    
    # 2. FIXER LA GRAINE
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        
    # 3. CRÉATION DU MODÈLE
    model = ProteinBiLSTM().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss() 
    
    # Variables locales au trial
    best_val_loss_trial = float('inf')
    best_epoch_trial = 0   # <--- AJOUT : Pour retenir l'époque
    patience_counter = 0
    best_model_trial_state = None 
    
    # --- ENTRAÎNEMENT (Full Batch) ---
    for epoch in range(MAX_EPOCHS):
        model.train()
        
        # Forward
        y_train = model(X_true)
        loss = loss_fn(y_train, y_true)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # --- VALIDATION & EARLY STOPPING ---
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                y_val_logits = model(X_val)
                val_loss = loss_fn(y_val_logits, y_val).item()
                
                # Si la loss val descend, on est contents
                if val_loss < best_val_loss_trial:
                    best_val_loss_trial = val_loss
                    best_epoch_trial = epoch  # <--- AJOUT : On capture l'époque
                    patience_counter = 0
                    best_model_trial_state = copy.deepcopy(model.state_dict())
                else:
                    patience_counter += 1
            
            # Si le modèle ne progresse plus en Val, on arrête ce Trial
            if patience_counter >= PATIENCE:
                break
"""

'\nimport time\nimport random\nimport copy\nimport numpy as np\nimport torch\nimport torch.nn as nn\n\n# --- CONFIGURATION DE LA BATAILLE ---\nNUM_TRIALS = 50          # 50 Combattants\nMAX_EPOCHS = 600         \nPATIENCE = 30            # Si la validation ne bouge pas pendant 30 tours -> Next\n\nglobal_best_model_state = None\nglobal_best_params = {}\n\nprint(f" DÉBUT DE LA BATTLE ROYALE : {NUM_TRIALS} COMBATTANTS (Objectif: Max Test Score)")\nprint("="*60)\n\nstart_time = time.time()\n\nfor trial in range(1, NUM_TRIALS + 1):\n    \n    # 1. PARAMÈTRES ALÉATOIRES\n    seed = random.randint(0, 10000)\n    # Learning Rate (Log-Uniform : explore aussi bien 0.0001 que 0.01)\n    lr_log = np.random.uniform(-4, -2.3) \n    lr = 10 ** lr_log\n    \n    # 2. FIXER LA GRAINE\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n        \n    # 3. CRÉATION DU MODÈLE\n    model = ProteinBiLSTM()

## Summary

| Model | Test Accuracy |
|-------|---------------|
| MLP (One-Hot) | 63,72% |
| MLP + Filter (Stage 2) | 65,20% |
| BiLSTM | 64,09% |
